# Chapter 3: Build the smallest useful runtime

Companion to *Agent Harness Engineering*. Listings 3.1 to 3.8 live in `ahe/ch03.py`.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. Fixed program versus planner (Table 3.1)

In [ ]:
from ahe import ch03
from ahe.ch03 import request, run_task, fixed_screen, fake_planner, packet
for pid in ("P041", "P043"):
    fixed = fixed_screen(request(pid, 3))
    planned = run_task(request(pid, 3), fake_planner)
    print(pid, "fixed:", fixed.results["no_anticoag"].status, "| planner:",
          planned.results["no_anticoag"].status, "reads", planned.reads)

## 2. Misbehaving planners (Table 3.3)

In [ ]:
for row in ch03.trials(request("P041", 3)):
    print(f"{row[0]:14s} {row[1]:10s} {row[2][:40]:40s} steps={row[3]:2d} reads={row[4]}")

## 3. One run's raw events

In [ ]:
run = run_task(request("P041", 3), fake_planner)
for e in run.events:
    print(e)
print(packet(run))

## 4. Replay the recorded live-planner runs (Table 3.5)

The runtime is deterministic given the planner's replies, so feeding the recorded replies back reproduces each run exactly.

In [ ]:
from ahe import ch01, live
recs = live.records("ch03_planner_gpt-6-luna")
replies = iter(r["reply"] for r in recs)
def replay(context):
    try:
        return ch01.parse_json(next(replies))
    except ValueError:
        return {"tool": None}
for pid in ["P041"] * 5 + ["P043"] * 5:
    r = run_task(request(pid, 3), replay)
    print(pid, r.state, r.count("proposal"), "steps,", r.reads, "reads,",
          r.results.get("no_anticoag", ("-",))[0])

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    ch01.chat = live.Recorder("my_ch03_planner")
    r = run_task(request("P043", 3), ch03.live_planner)
    for e in r.events:
        if e["kind"] == "proposal":
            print(e["proposal"])
    print(r.state, r.reason)
else:
    print("Set AHE_LIVE=1 to run the live planner.")

## Exercise solutions

### Exercise 3.1

A rule: when the step budget is spent and `unfinished(run)` is empty, complete the run with the reason "budget spent; every rule current". It is wrong when finishing is itself a decision the planner owes, for example when a later chapter requires a drafted packet or an approval before completion.

In [ ]:
def drive_autofinish(run, planner, reader=ch03.fake_reader):
    run = ch03.drive(run, planner, reader)
    if run.state == "blocked" and run.reason == "step budget spent" and not ch03.unfinished(run):
        run.state, run.reason = "completed", "budget spent; every rule current"
        run.record("stop", state=run.state, reason=run.reason)
    return run

### Exercise 3.2

The gate needs `draft_packet` in `SPEC` with no arguments. The completion check then also requires that a draft exists and that it was drafted after the last change to any rule's evidence.

In [ ]:
def t_draft(run):
    run.draft = packet(run)
    run.draft_seq = len(run.events)
    return {"drafted": sorted(run.draft["criteria"])}
ch03.TOOLS["draft_packet"] = t_draft
ch03.SPEC["draft_packet"] = {}
print(ch03.gate(ch03.start(request("P041", 3)), {"tool": "draft_packet", "args": {}}))
del ch03.TOOLS["draft_packet"], ch03.SPEC["draft_packet"]

### Exercise 3.3

Read the proposals the replay prints in section 4. Letting the model choose search terms unwatched is reasonable, because the gate bounds them and `ground` checks what they find. Stopping early is the choice to fixture first: 2 of the 10 recorded runs spent their budget with every rule current.